In [2]:
import astropy
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.io import fits
from astroquery.vizier import Vizier
import astroquery
from astroquery.gaia import Gaia

import pyvo as vo

Status messages could not be retrieved


In [34]:
tap_20 = vo.dal.TAPService('http://cda.cfa.harvard.edu/csc21tap')

In [35]:
for x in tap_20.tables.keys():
    print(x)

TAP_SCHEMA.schemas
TAP_SCHEMA.tables
TAP_SCHEMA.columns
TAP_SCHEMA.keys
TAP_SCHEMA.key_columns
csc21.cone
csc21.master_source
csc21.stack_source
csc21.observation_source
csc21.master_stack_assoc
csc21.stack_observation_assoc
csc21.detect_stack
csc21.valid_stack
csc21.likely_stack
csc21.image
ivoa.ObsCore


In [ ]:
for x in tap_20.tables:
    print(x)

<VODataServiceTable name="TAP_SCHEMA.schemas">... 4 columns ...</VODataServiceTable>
<VODataServiceTable name="TAP_SCHEMA.tables">... 6 columns ...</VODataServiceTable>
<VODataServiceTable name="TAP_SCHEMA.columns">... 13 columns ...</VODataServiceTable>
<VODataServiceTable name="TAP_SCHEMA.keys">... 6 columns ...</VODataServiceTable>
<VODataServiceTable name="TAP_SCHEMA.key_columns">... 4 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.master_source">... 515 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.stack_source">... 321 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.observation_source">... 779 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.master_stack_assoc">... 4 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.stack_observation_assoc">... 5 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.detect_stack">... 4 columns ...</VODataServiceTable>
<VODataServiceTable name="csc2.valid_sta

In [23]:
def _query_tap_columns(
    endpoint: str, search_term: str, table_name: str | None,
) -> list[dict[str, str]]:
    """Query the TAP service's TAP_SCHEMA for matching columns."""
    service = vo.dal.TAPService(endpoint)
    term = search_term.replace("'", "''")  # Escape for SQL

    where_parts = [
        f"(LOWER(column_name) LIKE '%{term.lower()}%'"
        f" OR LOWER(description) LIKE '%{term.lower()}%')",
    ]
    if table_name:
        safe_table = table_name.replace("'", "''")
        where_parts.append(f"LOWER(table_name) LIKE '%{safe_table.lower()}%'")

    adql = (
        "SELECT TOP 30 table_name, column_name, datatype, description, unit "
        "FROM TAP_SCHEMA.columns WHERE "
        + " AND ".join(where_parts)
        + " ORDER BY table_name, column_name"
    )
    result = service.search(adql)
    df = result.to_table().to_pandas()
    return [
        {
            "name": str(row.get("column_name", "")),
            "table": str(row.get("table_name", "")),
            "type": str(row.get("datatype", "")),
            "unit": str(row.get("unit", "") or ""),
            "description": str(row.get("description", "") or ""),
        }
        for _, row in df.iterrows()
    ]

print(_query_tap_columns('http://cda.cfa.harvard.edu/csc2tap'))

TypeError: _query_tap_columns() missing 2 required positional arguments: 'search_term' and 'table_name'

In [27]:
import pyvo as vo

tap_20 = vo.dal.TAPService("http://cda.cfa.harvard.edu/csc21tap")

def get_tap_metadata(tap):
    # all fully-qualified table names
    table_names = list(tap.tables.keys())

    # schema names inferred from fully-qualified table names like "csc2.master_source"
    schema_names = sorted({name.split(".", 1)[0] for name in table_names if "." in name})

    # all columns grouped by table
    columns_by_table = {
        table_name: [col.name for col in tap.tables[table_name].columns]
        for table_name in table_names
    }

    # optional: one flat unique list of all column names across all tables
    all_column_names = sorted({
        col.name
        for table_name in table_names
        for col in tap.tables[table_name].columns
    })

    return {
        "schemas": schema_names,
        "tables": table_names,
        "columns_by_table": columns_by_table,
        "all_column_names": all_column_names,
    }

get_tap_metadata(tap_20)

{'schemas': ['TAP_SCHEMA', 'csc21', 'ivoa'],
 'tables': ['TAP_SCHEMA.schemas',
  'TAP_SCHEMA.tables',
  'TAP_SCHEMA.columns',
  'TAP_SCHEMA.keys',
  'TAP_SCHEMA.key_columns',
  'csc21.cone',
  'csc21.master_source',
  'csc21.stack_source',
  'csc21.observation_source',
  'csc21.master_stack_assoc',
  'csc21.stack_observation_assoc',
  'csc21.detect_stack',
  'csc21.valid_stack',
  'csc21.likely_stack',
  'csc21.image',
  'ivoa.ObsCore'],
 'columns_by_table': {'TAP_SCHEMA.schemas': ['schema_name',
   'description',
   'utype',
   'id'],
  'TAP_SCHEMA.tables': ['schema_name',
   'table_name',
   'table_type',
   'description',
   'utype',
   'id'],
  'TAP_SCHEMA.columns': ['table_name',
   'column_name',
   'description',
   'unit',
   'ucd',
   'utype',
   'datatype',
   '"size"',
   'principal',
   'indexed',
   'std',
   'id',
   'xtype'],
  'TAP_SCHEMA.keys': ['key_id',
   'from_table',
   'target_table',
   'description',
   'utype',
   'id'],
  'TAP_SCHEMA.key_columns': ['key_id', 

In [41]:
import pyvo as vo

tap_20 = vo.dal.TAPService("http://cda.cfa.harvard.edu/csc21tap")

CSC2_TABLES = [
    "csc21.observation_source",
]

def get_columns(tap, tables=CSC2_TABLES, flatten=False):
    """
    Return column names for the selected TAP tables.

    Parameters
    ----------
    tap : pyvo.dal.TAPService
    tables : list[str]
    flatten : bool
        If True, also return one deduplicated flat list of all column names.

    Returns
    -------
    dict
    """
    available_tables = set(tap.tables.keys())

    found = [t for t in tables if t in available_tables]
    missing = [t for t in tables if t not in available_tables]

    columns_by_table = {
        table_name: [col.name for col in tap.tables[table_name].columns]
        for table_name in found
    }

    result = {
        "tables_found": found,
        "tables_missing": missing,
        "columns_by_table": columns_by_table,
    }

    if flatten:
        result["all_unique_columns"] = sorted({
            col
            for cols in columns_by_table.values()
            for col in cols
        })

    return result

In [42]:
cols = get_columns(tap_20)
cols


{'tables_found': ['csc21.observation_source'],
 'tables_missing': [],
 'columns_by_table': {'csc21.observation_source': ['obsid',
   'obi',
   'targname',
   'ra_targ',
   'dec_targ',
   'ra_pnt',
   'dec_pnt',
   'roll_pnt',
   'chipx_pnt',
   'chipy_pnt',
   'chip_id_pnt',
   'ra_nom',
   'dec_nom',
   'roll_nom',
   'sim_x',
   'sim_z',
   'dy',
   'dz',
   'dtheta',
   'deltax',
   'deltay',
   'deltarot',
   'dscale',
   'man_astrom_flag',
   'gti_start',
   'gti_stop',
   'gti_elapse',
   'gti_obs',
   'gti_end',
   'gti_mjd_obs',
   'mjd_ref',
   'instrument',
   'grating',
   'datamode',
   'readmode',
   'cycle',
   'exptime',
   'timing_mode',
   'ascdsver',
   'caldbver',
   'crdate',
   'ao',
   'region_id',
   'theta',
   'phi',
   'chipx',
   'chipy',
   'chip_id',
   'likelihood_b',
   'likelihood_h',
   'likelihood_m',
   'likelihood_s',
   'likelihood_u',
   'likelihood_w',
   'flux_significance_b',
   'flux_significance_h',
   'flux_significance_m',
   'flux_significa

In [44]:
def get_column_metadata(tap, tables=CSC2_TABLES):
    """
    Return detailed metadata for columns in the selected tables.

    Output format:
    {
        "csc2.master_source": [
            {
                "column_name": ...,
                "datatype": ...,
                "unit": ...,
                "ucd": ...,
                "utype": ...,
                "description": ...,
                "indexed": ...,
                "principal": ...,
                "std": ...
            },
            ...
        ],
        ...
    }
    """
    table_list = ", ".join(f"'{t}'" for t in tables)

    query = f"""
    SELECT
        table_name,
        column_name,
        datatype,
        unit,
        ucd,
        utype,
        description,
        indexed,
        principal,
        std
    FROM TAP_SCHEMA.columns
    WHERE table_name IN ({table_list})
    ORDER BY table_name, column_name
    """

    results = tap.search(query)

    metadata = {t: [] for t in tables}
    for row in results:
        metadata[row["table_name"]].append({
            "column_name": row["column_name"],
            "datatype": row["datatype"],
            "unit": row["unit"],
            "ucd": row["ucd"],
            "utype": row["utype"],
            "description": row["description"],
            "indexed": row["indexed"],
            "principal": row["principal"],
            "std": row["std"],
        })

    return metadata

get_column_metadata(tap_20, CSC2_TABLES)

{'csc21.observation_source': [{'column_name': 'ao',
   'datatype': 'INTEGER',
   'unit': '',
   'ucd': '',
   'utype': '',
   'description': 'Chandra observing cycle in which the observation was scheduled',
   'indexed': np.int32(0),
   'principal': np.int32(0),
   'std': np.int32(0)},
  {'column_name': 'apec_abund',
   'datatype': 'DOUBLE',
   'unit': '',
   'ucd': '',
   'utype': '',
   'description': 'Abundance of the best fitting  absorbed APEC model spectrum to the source region aperture PI spectrum',
   'indexed': np.int32(0),
   'principal': np.int32(0),
   'std': np.int32(0)},
  {'column_name': 'apec_abund_hilim',
   'datatype': 'DOUBLE',
   'unit': '',
   'ucd': '',
   'utype': '',
   'description': 'Abundance of the best fitting  absorbed APEC model spectrum to the source region aperture PI spectrum (68% upper confidence limit)',
   'indexed': np.int32(0),
   'principal': np.int32(0),
   'std': np.int32(0)},
  {'column_name': 'apec_abund_lolim',
   'datatype': 'DOUBLE',
   'u